# 기말 프로젝트 트랙 2 시작 노트북. DDIM 의 걸음 수
*Final project, Track 2 starter. DDIM and the number of sampling steps*

**확인할 가설 / Hypothesis to check.** 25 DDIM 걸음이면 표본이 100 걸음만큼 자료에 가깝다.
*25 DDIM steps give samples about as close to the data as 100 steps.*

9주차 실습과 같은 망 · 잡음 일정 · 학습입니다. 기준 설정은 실습 그대로(네 군집, 2,000회 학습)이고, 여러분은 **조건 하나**(자료 모양 또는 학습 길이)를 바꿔 같은 가설을 다시 봅니다. 망, 잡음 일정, 시작 잡음은 그대로 둡니다. 학습은 시드 셋(29 · 30 · 31)으로 자동 반복합니다. 2차원이라 한 번에 몇 초입니다. **가설이 맞지 않아도 근거를 갖춰 설명하면 같은 점수입니다.**
*The Week 9 lab's network, noise schedule and training. The baseline is the lab setting (four clusters, 2,000 updates); you change **one condition** (data shape or training length) and test the same hypothesis again, keeping the network, the noise schedule and the starting noises fixed. Training repeats over three seeds automatically; each run takes seconds. **A hypothesis that does not hold, explained with evidence, earns the same marks.***

**쓰는 법 / How to use.** 「내 선택」 칸에서 조건과 값을 고르고 작성칸 1 을 쓴 뒤 런타임 > 모두 실행 → 끝의 작성칸 2 ~ 4 와 AI 사용 칸에 답. 고르기 전에는 기준 설정만 돕니다. 전체 1분 안팎(CPU).
*Choose a condition and value in "Your choice", write answer 1, then Runtime > Run all, and fill answer cells 2 to 4 and AI use at the end. Before you choose, only the baseline runs. About a minute on a CPU.*

In [ ]:
#@title 준비 / Setup { display-mode: "form" }
import math, time, json, hashlib, platform, datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt
NOTEBOOK = "FP-Track2 v7"

torch.set_num_threads(2)                      # 실습과 같음. 같은 컴퓨터에서 다시 돌리면 같은 숫자 / as in the labs
GREEN, GOLD, RED, GRAY = "#406551", "#C9971C", "#B04A3A", "#9AA5A0"
T = 100
betas = torch.cat([torch.zeros(1), torch.linspace(0.0008, 0.08, T)])
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)

def t_features(t):
    tt = (t.float()[:, None] - 1) / T
    freqs = (2 ** torch.arange(8).float())[None, :]
    return torch.cat([torch.sin(tt * freqs * math.pi), torch.cos(tt * freqs * math.pi)], 1)

def jump(x0, eps, abar_t):
    return torch.sqrt(abar_t) * x0 + torch.sqrt(1 - abar_t) * eps

class Net(nn.Module):
    """9주차 실습의 잡음 예측기 / the Week 9 noise predictor (input 2 + 16, hidden 64 × 2)."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(18, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, t_features(t)], 1))

def make_four(gen):
    """9주차 실습: (±2, ±2) 네 군집, 흩어짐 0.24, 2,000점 / the Week 9 lab data."""
    C = torch.tensor([[-2., -2.], [-2., 2.], [2., -2.], [2., 2.]])
    ids = torch.randint(0, 4, (2000,), generator=gen)
    return C[ids] + torch.randn(2000, 2, generator=gen) * 0.24, C, None

def make_ring(gen):
    """과제 3: 반지름 2.5 고리 위 여섯 군집, 흩어짐 0.2, 위쪽 군집이 4% / the Assignment 3 data; the top cluster holds 4%."""
    A = torch.tensor([90., 150., 210., 270., 330., 30.]) * math.pi / 180
    C = torch.stack([2.5 * torch.cos(A), 2.5 * torch.sin(A)], 1)
    ids = torch.multinomial(torch.tensor([0.04] + [0.96 / 5] * 5), 2000, replacement=True, generator=gen)
    return C[ids] + torch.randn(2000, 2, generator=gen) * 0.2, C, 0

DATASETS = {"four clusters (Week 9 lab)": make_four, "six clusters on a ring, one rare (Assignment 3)": make_ring}

def train(data, updates, seed):
    """9주차 실습 §3 의 학습: 배치 512, 걸음 t 무작위, Adam 1e-3 / the lab's training."""
    gen = torch.Generator().manual_seed(seed); torch.manual_seed(seed)
    model = Net(); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(updates):
        idx = torch.randint(0, len(data), (512,), generator=gen); x0 = data[idx]
        t = torch.randint(1, T + 1, (len(x0),), generator=gen); eps = torch.randn(x0.shape, generator=gen)
        loss = (model(jump(x0, eps, abar[t][:, None]), t) - eps).square().mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def sample_ddim(model, x_T, steps):
    """9주차 실습 §5 의 DDIM: 새 잡음 없이 steps 걸음만 / the lab's DDIM: no new noise, only `steps` steps."""
    x = x_T.clone()
    ts = np.unique(np.round(np.linspace(T - 1, 0, steps)).astype(int))[::-1] + 1
    for j, t in enumerate(ts):
        s = int(ts[j + 1]) if j + 1 < len(ts) else 0
        eps_hat = model(x, torch.full((len(x),), int(t)))
        x0_hat = (x - torch.sqrt(1 - abar[t]) * eps_hat) / torch.sqrt(abar[t])
        x = torch.sqrt(abar[s]) * x0_hat + torch.sqrt(1 - abar[s]) * eps_hat
    return x

def measure(pts, centers, rare):
    """세 지표 / three measures.
    near_%         : 어느 중심에서든 1.0 안에 든 표본의 비율(%). 9주차 실습의 「중심 1.0 안」 / share within 1.0 of some center, as in the Week 9 lab
    cluster_spread : 퍼짐. 모든 표본을 가장 가까운 중심으로 나누고, 군집마다 두 좌표의 표준편차를 평균한 뒤 군집끼리 평균.
                     9주차 덱 「What Happens When We Skip Steps?」 의 spread 와 같은 계산이고, 거리로 거르지 않는다
                     spread: every sample goes to its nearest center; per cluster, average the standard deviations of the two
                     coordinates, then average over clusters. The spread on the Week 9 deck; no distance filter
    rare_%         : 드문 군집 중심에서 1.0 안에 든 표본의 비율(%, 고리 자료만) / share within 1.0 of the rare center (ring data only)"""
    d, which = torch.cdist(pts, centers).min(1); near = d < 1.0          # 가장 가까운 중심까지의 거리와 그 번호 / distance to, and index of, the nearest center
    sp = [pts[which == k].std(0, correction=0).mean().item()            # 그 중심이 가장 가까운 표본 전부. 덱처럼 n 으로 나눈 표준편차 / all samples nearest to it; std divided by n, as on the deck
          for k in range(len(centers)) if (which == k).sum() > 5]       # 다섯 점 이하인 군집은 재지 않음 / clusters with five points or fewer are not measured
    rare_share = 100 * (near & (which == rare)).float().mean().item() if rare is not None else float("nan")
    return 100 * near.float().mean().item(), float(np.mean(sp)), rare_share
print("준비 완료 / Setup ready | torch", torch.__version__)

## 내 선택 / Your choice
조건 하나만 바꿉니다. 기준은 「네 군집, 2,000회」입니다. *Change one condition. The baseline is four clusters, 2,000 updates.*

| 조건 / condition | 기준 / baseline | 바꿀 수 있는 값 / new value |
|---|---|---|
| data shape | four clusters (Week 9 lab) | six clusters on a ring, one rare (Assignment 3) |
| training length | 2,000 updates | 600 or 6,000 updates |

걸음 수 5 · 10 · 25 · 50 · 100, 시작 잡음 600개(시드 7), 학습 시드 29 · 30 · 31 은 고정입니다. 시드 29 의 기준 결과는 9주차 실습과 덱의 숫자와 같습니다(요약 셀의 CHECK 줄).
*Steps 5, 10, 25, 50, 100, the 600 starting noises (seed 7) and training seeds 29, 30, 31 are fixed. Seed 29 of the baseline reproduces the numbers of the Week 9 lab and deck (the CHECK line in the summary cell).*

| 지표 / measure | 뜻 / meaning |
|---|---|
| `near_%` | 어느 중심에서든 1.0 안에 든 표본의 비율. 9주차 실습의 「중심 1.0 안」<br>The share of samples within 1.0 of some center, as in the Week 9 lab |
| `cluster_spread` | 퍼짐. 모든 표본을 가장 가까운 중심으로 나눠 군집마다 잰 표준편차의 평균. 9주차 덱 「What Happens When We Skip Steps?」 의 spread 이고, 군집 사이로 흘린 표본도 넣습니다<br>Spread: every sample goes to its nearest center, and the clusters' standard deviations are averaged. The spread on the Week 9 slide; samples spilled between clusters count too |
| `rare_%` | 드문 군집 중심에서 1.0 안에 든 표본의 비율. 고리 자료만<br>The share of samples within 1.0 of the rare center. Ring data only |

In [ ]:
#@title 내 선택 / Your choice { display-mode: "form" }
CONDITION = "choose one"  #@param ["choose one", "data shape", "training length"]
NEW_DATA_SHAPE = "six clusters on a ring, one rare (Assignment 3)"  #@param ["six clusters on a ring, one rare (Assignment 3)"]
NEW_TRAINING_UPDATES = 6000  #@param [600, 6000] {type:"raw"}

BASELINE = {"data": "four clusters (Week 9 lab)", "updates": 2000}
CHANGED = dict(BASELINE)
if CONDITION == "data shape": CHANGED["data"] = NEW_DATA_SHAPE
if CONDITION == "training length": CHANGED["updates"] = int(NEW_TRAINING_UPDATES)
SETTINGS = [("baseline", BASELINE)] + ([("changed", CHANGED)] if CONDITION != "choose one" else [])
CHOICE = {"condition": CONDITION, "baseline": BASELINE, "changed": CHANGED if CONDITION != "choose one" else None}
for name, s in SETTINGS: print(f"{name:9s}: {s['data']} | {s['updates']} updates")
if CONDITION == "choose one":
    print("아직 조건을 고르지 않았습니다. 기준 설정만 돌립니다 / No condition chosen yet: only the baseline runs.")

## 작성칸 1. 비교 목적과 판정 기준 / Answer 1. Purpose and decision rule
「내 선택」 을 정한 뒤, 그 설정으로 돌리기 전에 아래 칸에 쓰세요. 제안서의 평가 계획을 옮겨 와도 됩니다. 결과를 본 뒤에 고치지 말고, 결과를 본 뒤에 썼다면 그렇다고 밝히세요.
*Write this in the cell below after choosing and before you run your choice; you may carry over the evaluation plan from your proposal. Do not revise it after seeing results, and if you wrote it after seeing results, say so.*

- **목적 / Purpose.** 바꾼 조건과 값, 고정한 것, 이 비교로 답하려는 질문. 제안서와 달라진 설정이 있으면 무엇이 왜 달라졌는지.
  *The condition and values you change, what stays fixed, and the question this comparison answers; if any setting differs from your proposal, what changed and why.*
- **예측 / Prediction.** 지표마다 어느 쪽으로 움직일지와 그렇게 보는 이유.
  *For each measure, which way it will move and why you think so.*
- **판정 기준 / Decision rule.** 「25 걸음이면 100 걸음만큼 가깝다」 를 어느 지표로 보고, 25 걸음과 100 걸음의 차이가 얼마 안이면 「만큼」 으로 볼지.
  *Which measure decides "25 steps are about as close as 100 steps", and how small the difference between 25 and 100 steps must be to count as "about as close".*

**답 / Answer.** 여기에 쓰세요. *Write here.*

In [ ]:
#@title 실행 / Run. 1분 안팎 / about a minute { display-mode: "form" }
STEPS, SEEDS = (5, 10, 25, 50, 100), (29, 30, 31)
x_T = torch.randn(600, 2, generator=torch.Generator().manual_seed(7))      # 모든 설정 · 걸음에 같은 시작 잡음 / the same starting noises everywhere
rows, examples, t0 = [], {}, time.time()
for label, s in SETTINGS:
    data, centers, rare = DATASETS[s["data"]](torch.Generator().manual_seed(29))
    nr, sp, ra = measure(data[:600], centers, rare)
    rows.append({"setting": label, "seed": "real data", "steps": 0, "near_%": nr, "cluster_spread": sp, "rare_%": ra})
    examples[(label, "real")] = (data[:600], centers, rare)
    for seed in SEEDS:
        model = train(data, s["updates"], seed)
        for k in STEPS:
            pts = sample_ddim(model, x_T, k)
            nr, sp, ra = measure(pts, centers, rare)
            rows.append({"setting": label, "seed": seed, "steps": k, "near_%": nr, "cluster_spread": sp, "rare_%": ra})
            if seed == SEEDS[0]: examples[(label, k)] = (pts, centers, rare)
RESULTS = pd.DataFrame(rows)
RESULTS.to_csv("track2_results.csv", index=False)
print(f"완료 / done in {time.time() - t0:.0f} s. 결과는 track2_results.csv 에도 저장 / also saved to track2_results.csv")

In [ ]:
#@title 요약 / Summary { display-mode: "form" }
gen_rows = RESULTS[RESULTS["seed"] != "real data"]
agg = lambda c: gen_rows.groupby(["setting", "steps"])[c].agg(lambda v: "-" if v.isna().all() else f"{v.mean():.2f} ({v.min():.2f}–{v.max():.2f})")
cols = ["near_%", "cluster_spread"] + (["rare_%"] if gen_rows["rare_%"].notna().any() else [])
summary = pd.concat([agg(c) for c in cols], axis=1); summary.columns = [c + " mean (min–max over 3 seeds)" for c in cols]
print("SUMMARY. 걸음 수별 평균과 시드 범위 / mean and seed range per step count"); print(summary.to_string()); print()
real = RESULTS[RESULTS["seed"] == "real data"][["setting"] + cols]
print("REAL DATA. 같은 지표를 진짜 자료 600점에서 / the same measures on 600 real points"); print(real.round(3).to_string(index=False)); print()
r = gen_rows.pivot_table(index=["setting", "seed"], columns="steps", values=cols)
check = pd.DataFrame({c + " (25 − 100)": r[c][25] - r[c][100] for c in cols})
print("HYPOTHESIS CHECK. 같은 망에서 25 걸음 값 − 100 걸음 값 / 25-step value minus 100-step value, same network"); print(check.round(3).to_string()); print()
b29 = gen_rows[(gen_rows["setting"] == "baseline") & (gen_rows["seed"] == 29)].set_index("steps")[["near_%", "cluster_spread"]]
print("CHECK. 기준 설정 · 시드 29 = 9주차 실습과 덱 / baseline, seed 29 = the Week 9 lab and deck"); print(b29.T.round(2).to_string())
# 기대값 / Expected   (5 · 10 · 25 · 100 걸음은 9주차 실습 §5 와 덱 「What Happens When We Skip Steps?」 의 숫자 / the Week 9 lab and slide)
#   steps              5      10      25      50     100
#   near_%         79.17   90.17   94.17   95.67   96.00
#   cluster_spread  0.51    0.37    0.33    0.32    0.32
#   진짜 자료(위 REAL DATA 표)의 퍼짐은 약 0.24 입니다. 다른 줄이 나오면 위 셀을 바꾸지 않았는지 보세요(torch 판에 따라 끝자리는 다를 수 있음).
#   The real data's spread (REAL DATA table above) is about 0.24. Other numbers mean a cell above was changed (the last digit can vary with the torch version).

In [ ]:
#@title 그림 / Figures { display-mode: "form" }
for label, _ in SETTINGS:
    fig, axs = plt.subplots(1, len(STEPS) + 1, figsize=(2.2 * (len(STEPS) + 1), 2.5))
    for ax, k in zip(axs, list(STEPS) + ["real"]):
        pts, centers, rare = examples[(label, k)]
        ax.scatter(pts[:, 0], pts[:, 1], s=2, color=GRAY if k == "real" else GREEN, alpha=0.5)
        ax.scatter(centers[:, 0], centers[:, 1], marker="+", s=50, color=GOLD, linewidths=1.3)
        if rare is not None: ax.scatter(centers[rare:rare + 1, 0], centers[rare:rare + 1, 1], marker="+", s=80, color=RED, linewidths=1.8)
        ax.set(xlim=(-4, 4), ylim=(-4, 4), xticks=[], yticks=[], title="real data" if k == "real" else f"{k} steps"); ax.set_aspect("equal")
    plt.suptitle(f"{label}: DDIM samples from seed 29, same 600 starting noises", fontsize=10); plt.tight_layout(); plt.show()
fig, axs = plt.subplots(1, len(cols), figsize=(4.2 * len(cols), 3.0))
for ax, c in zip(np.atleast_1d(axs), cols):
    for label, col in zip([s[0] for s in SETTINGS], (GREEN, RED)):
        part = gen_rows[gen_rows["setting"] == label]
        for seed in SEEDS:
            p = part[part["seed"] == seed]; ax.plot(p["steps"], p[c], color=col, alpha=0.35, lw=1)
        m = part.groupby("steps")[c].mean(); ax.plot(m.index, m.values, color=col, lw=2.2, marker="o", label=label)
        rv = RESULTS[(RESULTS["setting"] == label) & (RESULTS["seed"] == "real data")][c].iloc[0]
        if not np.isnan(rv): ax.axhline(rv, color=col, ls=":", lw=1)
    ax.set(xscale="log", xticks=list(STEPS), xticklabels=[str(s) for s in STEPS], xlabel="DDIM steps (network passes)", title=c)
    ax.legend(fontsize=8, frameon=False)
plt.suptitle("Thin lines: each training seed. Thick: mean. Dotted: real data.", fontsize=10); plt.tight_layout(); plt.show()

In [ ]:
#@title 기록 / Record. 출력을 지우지 마세요 / keep this output { display-mode: "form" }
# 결과 해시는 같은 결과인지 대조하는 용도입니다. 실행 시간은 빼고 셉니다 / the results hash is for comparing results; run times are left out
record = {"notebook": NOTEBOOK, "choice": CHOICE,
          "results_sha256": hashlib.sha256(RESULTS.drop(columns=["seconds"], errors="ignore").to_csv(index=False).encode()).hexdigest()[:16],
          "python": platform.python_version(), "torch": torch.__version__, "device": "CPU (GPU available: " + ("yes" if torch.cuda.is_available() else "no") + ")",
          "run_at": datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}
if "RUN_EXTENSION" in globals():                                   # 트랙 3 의 선택 확장 / Track 3's optional extension
    record["extension"] = ({"p_drop": float(EXT_P_DROP), "results_sha256": hashlib.sha256(EXT_RESULTS.to_csv(index=False).encode()).hexdigest()[:16]}
                           if RUN_EXTENSION and "EXT_RESULTS" in globals() else "not run")
print(json.dumps(record, ensure_ascii=False, indent=2))

## 이 노트북이 보고서입니다 / This notebook is your report
아래 작성칸 2 ~ 4 와 AI 사용 칸에 답을 쓰고, 출력을 저장한 채 내세요. 숫자는 위 표에서 그대로 인용합니다. 위 기록 셀의 출력은 지우지 마세요(재현성 1점).
*Answer in cells 2 to 4 and the AI-use cell below, and submit with the outputs saved. Quote numbers from the tables above. Keep the output of the record cell above (reproducibility, 1 point).*

포스터에는 요약 표와 그림 하나를 옮기면 됩니다. 표와 그림을 옮긴 것만으로는 분석 점수를 받지 않습니다.
*For the poster, copy the summary table and one figure. Copying tables and figures alone does not earn the analysis marks.*

## 작성칸 2. 내 결과와 판정 / Answer 2. Your results and judgement
- **관찰 / Observation.** 표와 그림에서 본 것을 숫자로. 기준 설정과 바꾼 설정을 나란히.
  *What the tables and figures show, in numbers, with the baseline next to your changed setting.*
- **가능한 설명 / Possible explanation.** 왜 그렇게 나왔다고 보는지. 관찰과 구분하고, 확인하지 않은 설명이면 그렇다고 씁니다.
  *Why you think it came out this way, kept apart from the observation; say so if you have not tested the explanation.*
- **판정 / Judgement.** 작성칸 1 의 판정 기준으로 가설이 맞았는지. 가장 중요한 결론을, 그것을 받치는 숫자 둘과 그 숫자가 나온 설정으로 씁니다(필수 문단).
  *Did the hypothesis hold under your decision rule in answer 1? Give your main conclusion with the two numbers that support it and the setting they come from (the required paragraph).*
- 어느 지표를 먼저 봤는지, HYPOTHESIS CHECK 표의 25 − 100 차이가 얼마인지, 그 차이를 시드 사이의 변동(SUMMARY 의 범위)과 견주어 어떻게 읽었는지, 진짜 자료(REAL DATA 표)와는 어떤지 쓰세요. 통계 검정이나 추가 실행은 필요 없습니다.
  *Say which measure you looked at first, how large the 25 − 100 difference in the HYPOTHESIS CHECK table is, how you read it against the variation between seeds (the ranges in SUMMARY), and how it compares with the real data (the REAL DATA table). No formal test or extra runs are needed.*

**답 / Answer.** 여기에 쓰세요. *Write here.*

## 작성칸 3. 한계와 판단을 바꿀 증거 / Answer 3. Limits, and evidence that would change your judgement
- 이 실행이 확인한 것과 확인하지 못한 것. 예를 들어 표집 시드만 바꾼 것은 학습이 안정적인지를 확인하지 않습니다.
  *What these runs checked and what they did not; for example, changing only the sampling seed does not test whether training is stable.*
- 결론이 맞지 않을 수 있는 상황.
  *Situations where the conclusion may not hold.*
- 어떤 증거가 나오면 판단을 바꾸겠는지.
  *What evidence would make you change your judgement.*
- 노트북을 바꿨다면 무엇을 바꿨는지(예: 시드 추가).
  *Anything you changed in the notebook, such as adding seeds.*

**답 / Answer.** 여기에 쓰세요. *Write here.*

## 작성칸 4. 받은 질문과 답 / Answer 4. The question you received, and your answer
12/16 까지 포스터 댓글로 받은 질문 하나를 옮겨 적고, 이 노트북의 결과를 근거로 답하세요.
*Copy one question you received in the comments on your poster by 16 Dec, and answer it with evidence from this notebook.*

**답 / Answer.** 여기에 쓰세요. *Write here.*

## AI 사용 / AI use
도구, 용도, 대략의 범위를 쓰세요. 쓰지 않았으면 「없음」. 결론과 그 근거는 본인의 판단이어야 합니다.
*The tool, its purpose and the rough extent; write "none" if you used none. The conclusions and the evidence for them must be your own judgement.*

**답 / Answer.** 여기에 쓰세요. *Write here.*